In [1]:
import os
import glob
import pandas as pd
import geopandas as gpd

file_inputs = [
    {"path": "../data/alabama2026.geojson", "postal": "AL"},
    {"path": "../data/california2026.geojson", "postal": "CA"},
    {"path": "../data/florida2026.geojson", "postal": "FL"},
    {"path": "../data/louisiana2026.geojson", "postal": "LA"},
    {"path": "../data/northcarolina2026.geojson", "postal": "NC"},
    {"path": "../data/ohio2026.geojson", "postal": "OH"},
    {"path": "../data/tennessee2026.geojson", "postal": "TN"},
    {"path": "../data/texas2026.geojson", "postal": "TX"},
    {"path": "../data/utah2026.geojson", "postal": "UT"},
]

# Standard target CRS (EPSG:4326 is standard for GeoJSON)
TARGET_CRS = "EPSG:4326"
all_gdfs = []

for state in file_inputs:
    gdf = gpd.read_file(state["path"])

    if gdf.crs != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)
        
    # Create matching key for the geojson/csv files with state postal and district number
    gdf["District"] = state["postal"] + gdf["id"].astype(str).str.zfill(2)
    
    gdf = gdf[["id", "District", "DemPct", "RepPct", "Margin","geometry"]]
    
    all_gdfs.append(gdf)

states_gdf = pd.concat(all_gdfs, ignore_index=True)
states_gdf.head(8)

,id,District,DemPct,RepPct,Margin,geometry
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ..."
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ..."
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ..."
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ..."
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ..."
5,6,AL06,0.320528,0.662526,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ..."
6,7,AL07,0.582883,0.405147,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ..."
7,1,CA01,0.544906,0.422732,0.044906,"MULTIPOLYGON (((-121.32264 41.18387, -121.3308..."


In [2]:
# 2024 presidential results by Congressional district
pres24 = pd.read_csv("../data/2024_pres.csv")

# Merged desired columns from the csv file into the geojson file
columns_to_keep = ['District', 'Harris', 'Trump 24', 'Total']

pres24 = pres24[columns_to_keep]

# Inner join the datasets on matching District values
merged_gdf = states_gdf.merge(pres24, on='District', how='inner')
merged_gdf.head(8)

,id,District,DemPct,RepPct,Margin,geometry,Harris,Trump 24,Total
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",73003,257060,332700
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",155603,131721,290033
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",82654,229676,314869
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",53098,267953,323449
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",122344,224704,351650
5,6,AL06,0.320528,0.662526,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ...",105388,239984,349125
6,7,AL07,0.582883,0.405147,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ...",180321,111517,294526
7,1,CA01,0.544906,0.422732,0.044906,"MULTIPOLYGON (((-121.32264 41.18387, -121.3308...",118090,199649,326893


In [3]:
# rename columns
gdf_clean = merged_gdf.rename(columns={
    'id': 'District No.',
    'DemPct': 'Harris New',
    'RepPct': 'Trump New',
    'Margin': 'Partisan Index',
    'Harris': 'Harris 24 Raw',
    'Trump 24': 'Trump 24 Raw',
    'Total': 'Total 24 Raw'
})

gdf_clean.head(8)

,District No.,District,Harris New,Trump New,Partisan Index,geometry,Harris 24 Raw,Trump 24 Raw,Total 24 Raw
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",73003,257060,332700
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",155603,131721,290033
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",82654,229676,314869
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",53098,267953,323449
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",122344,224704,351650
5,6,AL06,0.320528,0.662526,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ...",105388,239984,349125
6,7,AL07,0.582883,0.405147,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ...",180321,111517,294526
7,1,CA01,0.544906,0.422732,0.044906,"MULTIPOLYGON (((-121.32264 41.18387, -121.3308...",118090,199649,326893


In [4]:
# Create new columns converting the raw votes into percentages, as well as show the percent margin difference
gdf_clean['Harris 24'] = gdf_clean['Harris 24 Raw'] / gdf_clean['Total 24 Raw']
gdf_clean['Trump 24'] = gdf_clean['Trump 24 Raw'] / gdf_clean['Total 24 Raw']

gdf_clean['Margin 24'] = gdf_clean['Harris 24'] - gdf_clean['Trump 24']
gdf_clean['Margin New'] = gdf_clean['Harris New'] - gdf_clean['Trump New']

gdf_clean['Margin Shift'] = gdf_clean['Margin New'] - gdf_clean['Margin 24']

# Remove the columns that we no longer need
# gdf_clean = gdf_clean.drop(columns=['Harris 24 Raw', 'Trump 24 Raw', 'Total 24 Raw'])

gdf_clean.head()

,District No.,District,Harris New,Trump New,Partisan Index,geometry,Harris 24 Raw,Trump 24 Raw,Total 24 Raw,Harris 24,Trump 24,Margin 24,Margin New,Margin Shift
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",73003,257060,332700,0.219426,0.772648,-0.553222,-0.358934,0.194288
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",155603,131721,290033,0.536501,0.454159,0.082342,-0.143004,-0.225346
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",82654,229676,314869,0.262503,0.729434,-0.466931,-0.465175,0.001756
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",53098,267953,323449,0.164162,0.828424,-0.664262,-0.663093,0.001169
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",122344,224704,351650,0.347914,0.638999,-0.291085,-0.289816,0.001269


In [5]:
# Implement column to flag districts targeted by redistricting
# 1. Base rule: Districts where winning party flipped
gdf_clean['Targeted'] = (gdf_clean['Margin New'] * gdf_clean['Margin 24']) < 0

# Explicitly ensure boolean dtype to prevent dtype warnings
gdf_clean['Targeted'] = gdf_clean['Targeted'].astype(bool)

# 2. Exclude CA09
gdf_clean.loc[gdf_clean['District'] == 'CA09', 'Targeted'] = False

# 3. Additional targeted districts
targeted_additions = ['NC01', 'OH09', 'TX28', 'TX34']
gdf_clean.loc[gdf_clean['District'].isin(targeted_additions), 'Targeted'] = True

# Reorder the columns
reordered_columns = ['District No.', 'District', 'Harris New', 'Trump New', 'Margin New',
                    'Margin Shift', 'Partisan Index', 'Harris 24', 'Trump 24', 'Margin 24', 
                    'Targeted', 'geometry']

gdf_clean = gdf_clean[reordered_columns]

gdf_clean.to_file("../data/map2026.geojson", driver='GeoJSON')
gdf_clean.head(16)

,District No.,District,Harris New,Trump New,Margin New,Margin Shift,Partisan Index,Harris 24,Trump 24,Margin 24,Targeted,geometry
0,1,AL01,0.314327,0.673261,-0.358934,0.194288,-0.185673,0.219426,0.772648,-0.553222,False,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ..."
1,2,AL02,0.422827,0.565831,-0.143004,-0.225346,-0.077173,0.536501,0.454159,0.082342,True,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ..."
2,3,AL03,0.261882,0.727057,-0.465175,0.001756,-0.238118,0.262503,0.729434,-0.466931,False,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ..."
3,4,AL04,0.163376,0.826469,-0.663093,0.001169,-0.336624,0.164162,0.828424,-0.664262,False,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ..."
4,5,AL05,0.346103,0.635919,-0.289816,0.001269,-0.153897,0.347914,0.638999,-0.291085,False,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ..."
5,6,AL06,0.320528,0.662526,-0.341998,0.043526,-0.179472,0.301863,0.687387,-0.385524,False,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ..."
6,7,AL07,0.582883,0.405147,0.177735,-0.055874,0.082883,0.612241,0.378632,0.233609,False,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ..."
7,1,CA01,0.544906,0.422732,0.122174,0.371672,0.044906,0.361250,0.610747,-0.249498,True,"MULTIPOLYGON (((-121.32264 41.18387, -121.3308..."
8,2,CA02,0.607539,0.361039,0.246500,-0.205085,0.107539,0.708815,0.257230,0.451585,False,"MULTIPOLYGON (((-121.88922 42.00324, -121.8913..."
9,3,CA03,0.533603,0.431771,0.101832,0.140019,0.033603,0.465207,0.503394,-0.038187,True,"MULTIPOLYGON (((-121.27450 38.51286, -121.2720..."
